In [1]:
import os
import pandas as pd
from collections import defaultdict

plants = os.listdir("automation/proteins")
info = pd.read_csv("string/9606.protein.info.v12.0.txt",sep='\t', encoding='utf-8')
data = pd.read_csv("string/9606.protein.links.v12.0.txt",sep=' ', encoding='utf-8')

# data = data[:500]
print(info.head())
print(data.head())
info_dict = dict(zip(info['preferred_name'], info['#string_protein_id']))
protein_link_dict = defaultdict(dict)

for p1, p2, score in zip(
    data['protein1'].to_numpy(),
    data['protein2'].to_numpy(),
    data['combined_score'].to_numpy()
):
    protein_link_dict[p1][p2] = score
print(len(protein_link_dict))



     #string_protein_id preferred_name  protein_size  \
0  9606.ENSP00000000233           ARF5           180   
1  9606.ENSP00000000412           M6PR           277   
2  9606.ENSP00000001008          FKBP4           459   
3  9606.ENSP00000001146        CYP26B1           512   
4  9606.ENSP00000002125        NDUFAF7           441   

                                          annotation  
0  ADP-ribosylation factor 5; GTP-binding protein...  
1  Cation-dependent mannose-6-phosphate receptor;...  
2  Peptidyl-prolyl cis-trans isomerase FKBP4, N-t...  
3  Cytochrome P450 26B1; Involved in the metaboli...  
4  Protein arginine methyltransferase NDUFAF7, mi...  
               protein1              protein2  combined_score
0  9606.ENSP00000000233  9606.ENSP00000356607             173
1  9606.ENSP00000000233  9606.ENSP00000427567             154
2  9606.ENSP00000000233  9606.ENSP00000253413             151
3  9606.ENSP00000000233  9606.ENSP00000493357             471
4  9606.ENSP00000000233

In [2]:
from sys import getsizeof

getsizeof(protein_link_dict)

415160

In [ ]:
import pickle
with open("protein_link_dict.pkl","wb") as file:
    pickle.dump(protein_link_dict,file)

In [ ]:
plants = os.listdir("automation\proteins")
print(plants)
for plant in plants:
    print(f"{plant} started")
    temp_protein_list = pd.read_csv(f"automation/proteins/{plant}")['Protein'].tolist()
    protein_list=[]
    for i in temp_protein_list:
        if i not in protein_list: protein_list.append(i)
    n = len(protein_list)
    df = pd.DataFrame(columns=["Protein1", "Protein2", "Score"])
    for i in range(n-1):
        try:
            protein_1 = protein_list[i]
            id1 = info_dict[protein_1]
            for j in range(i+1, n):
                try:
                    protein_2 = protein_list[j]
                    id2 = info_dict[protein_2]
                    # print(id1,id2)
                    score = protein_link_dict.get(id1, {}).get(id2)
                    if score:
                        # print(f"Success for protein {protein_2}")
                        score = score/1000
                        df.loc[len(df)] = [protein_1, protein_2, score]
                        # print(f"Score of {protein_1} and {protein_2} is {score}")
                except: pass
        except:
            pass
    df.to_csv(f"{plant.split('.')[0]}.csv",index=False)
    print(f"{plant} ended")

In [ ]:
checkp1=str(input())
checkp2=str(input())
checkid1 = info_dict[checkp1] 
checkid2 = info_dict[checkp2]
checkreq = data[(data['protein1'] == checkid1) & (data['protein2'] == checkid2)] 
score = int(checkreq['combined_score'].iloc[0]) / 1000
print(score)